# Nemotron-3 Omni LoRA Fine-Tuning on CORD-v2 with Megatron Bridge

This notebook walks you through LoRA PEFT fine-tuning of **Nemotron-3 Omni (33B MoE)** on the
**CORD-v2** receipt-parsing dataset using a single 8×H100 80 GiB node and the Megatron Bridge backend.

CORD-v2 is an image-only dataset (~800 train / 100 val / 100 test samples) that asks the model
to parse receipt images into structured XML-like token sequences. It is a good end-to-end
test for multimodal LoRA training on a single node.

For more examples, see https://github.com/NVIDIA-NeMo/Megatron-Bridge/tree/nemotron-3-omni/examples/models/vlm/nemotron_3_omni. 

---

## Prerequisites

- 8 × H100 80 GiB GPUs (single node)
- A HuggingFace model directory containing `config.json`, tokenizer, and processor and weights files


---
## Launching the Container

Pull and start the **NeMo 26.04** base image. A single host directory (`$WORKSPACE`) is
mounted at `/workspace` inside the container, containing all repos and model artifacts.

### Workspace layout

```
$WORKSPACE/                                          ← host dir → /workspace inside container
├── Megatron-Bridge/                                 ← cloned in Step 3 (training framework)
│   ├── scripts/training/run_recipe.py
│   ├── examples/
│   └── src/
├── Nemotron/                                        ← https://github.com/NVIDIA-NeMo/Nemotron
│   └── usage-cookbook/
│       └── Nemotron-3-Nano-Omni/
│           └── Megatron-bridge/
│               └── mbridge_lora_cord_v2_cookbook.ipynb  ← you are here
├── models/
│   ├── Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16/  ← HF model dir (config, tokenizer, processor)
│   └── megatron_base/                              ← Megatron checkpoint (Step 1, shared across datasets)
└── results/
    └── cord_v2/
        ├── checkpoints/                             ← LoRA training checkpoints (Steps 4–5)
        ├── eval_results/                           
        └── tensorboard/
```

### 1 — Set host-side variables

```bash
# Absolute path to your workspace directory on the host
export WORKSPACE="/path/to/your/workspace"

# HuggingFace dataset / model cache (shared across runs, kept outside the workspace)
export HF_HOME="${HOME}/.cache/huggingface"
```

### 2 — Start the container

```bash
CONTAINER="nvcr.io/nvidia/nemo:26.04.00"

docker container run \
  --gpus all -it --rm \
  --shm-size=16g --net=host --ipc=host \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  -e HF_TOKEN \
  -e HF_HOME=/root/.cache/huggingface \
  -v "${HF_HOME}:/root/.cache/huggingface" \
  -v "${WORKSPACE}:/workspace" \
  -w /workspace \
  "${CONTAINER}" bash
```

Mount summary:

| Host path | Container path | Contents |
|---|---|---|
| `$HF_HOME` | `/root/.cache/huggingface` | HF dataset + model cache |
| `$WORKSPACE` | `/workspace` | All repos, models, and results |

### 3 — Launch Jupyter

`Nemotron` is assumed to already exist at `/workspace/Nemotron/` — if you are reading
this notebook, it is already cloned. Run the following from inside the container:

```bash
cd /workspace/Nemotron

jupyter lab \
  --ip=0.0.0.0 \
  --port=8888 \
  --no-browser \
  --NotebookApp.token='' \
  --NotebookApp.password='' \
  usage-cookbook/Nemotron-3-Nano-Omni/Megatron-bridge/
```

Then open `http://<host-ip>:8888` in your browser and open `mbridge_lora_cord_v2_cookbook.ipynb`.

> If port 8888 is in use, pass `--port=8889` (or any free port) and update the URL.
>
> **Don't have Nemotron yet?** Clone it on the host before starting the container:
> ```bash
> cd $WORKSPACE
> git clone https://github.com/NVIDIA-NeMo/Nemotron.git
> ```


---
## Setup — Clone and Install Megatron-Bridge

**One-time setup** — skip if `/workspace/Megatron-Bridge` already exists (it persists on the
host through the volume mount and survives container restarts).


In [ ]:
%%bash
set -euo pipefail

if [ -d /workspace/Megatron-Bridge ]; then
    echo "Megatron-Bridge already exists — skipping clone."
else
    # Clone and check out the NemotronOmni branch
    cd /workspace
    git clone -b nemotron-3-omni https://github.com/NVIDIA-NeMo/Megatron-Bridge.git
    cd Megatron-Bridge
    git submodule update --init --recursive --depth 1
fi

# Install (idempotent — safe to re-run)
cd /workspace/Megatron-Bridge
uv lock
uv sync

# Verify
cd /workspace # Must execute from outside of the Megatron-Bridge directory
uv run python -c "
import megatron.core, megatron.bridge
print('core:  ', megatron.core.__path__)
print('bridge:', megatron.bridge.__path__)
"

---
## Setup — Get the HF Model

The model must be present at `/workspace/models/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16` before
running any training or conversion step. Use **one** of the two options below.

> **Skip both** if the directory already exists.

### Option A — Download from HuggingFace Hub

Requires a HuggingFace account with access to the model and `$HF_TOKEN` set in the
environment.

In [ ]:
%%bash
set -euo pipefail

HF_REPO="nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"
TARGET="/workspace/models/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"

if [ -e "$TARGET" ]; then
    echo "Already exists: $TARGET — skipping download."
    exit 0
fi

mkdir -p /workspace/models

export HF_TOKEN=<YOUR_HF_TOKEN>

huggingface-cli download \
    "$HF_REPO" \
    --local-dir "$TARGET" \
    --local-dir-use-symlinks False

echo ""
echo "Downloaded to: $TARGET"
ls "$TARGET"

### Option B — Symlink from the Host (before starting the container)

If the model already exists on the host filesystem (e.g. on a shared Lustre mount), create
a symlink inside `$WORKSPACE/models/` **on the host** before launching the container.
The container only sees paths under `/workspace`, so host paths outside `$WORKSPACE` are
not accessible from inside the container or from this notebook.

Run the following **on the host** (not in the notebook):

```bash
mkdir -p $WORKSPACE/models
ln -s /path/to/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 \
      $WORKSPACE/models/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16
```

Then verify the model is reachable from inside the container:

In [ ]:
%%bash
# Verify the model is visible inside the container
TARGET="/workspace/models/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"
if [ -e "$TARGET" ]; then
    echo "OK: $TARGET"
    ls "$TARGET" | head -5
else
    echo "NOT FOUND: $TARGET"
    echo "Create the symlink on the host first, then re-run this cell."
    exit 1
fi

---
### Model paths

Edit the two paths below to match your environment, then run the cell.

In [ ]:
import os

HF_MODEL_PATH = "/workspace/models/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"
MEGATRON_CKPT = "/workspace/models/megatron_base"

%env HF_MODEL_PATH=$HF_MODEL_PATH
%env MEGATRON_CKPT=$MEGATRON_CKPT
print(f"HF model : {HF_MODEL_PATH}")
print(f"Megatron : {MEGATRON_CKPT}")

---
## Step 1 — Model Conversion (HuggingFace → Megatron)

The Megatron backend expects model input in the Megatron format. Convert the HuggingFace checkpoint into a Megatron distributed checkpoint before training.
This is a one-time step; the result is reused for all subsequent training runs.

In [ ]:
%%bash
set -euo pipefail
cd /workspace/Megatron-Bridge

echo "=== Model Conversion: HuggingFace → Megatron ==="
echo "  Source (HF)   : $HF_MODEL_PATH"
echo "  Output (Megatron): $MEGATRON_CKPT"
echo ""

mkdir -p "$MEGATRON_CKPT"

uv run python examples/conversion/convert_checkpoints.py import \
    --hf-model       "$HF_MODEL_PATH" \
    --megatron-path  "$MEGATRON_CKPT" \
    --trust-remote-code

echo ""
echo "Conversion complete. Megatron checkpoint layout:"
ls "$MEGATRON_CKPT"

---
## Step 2 — Dataset

No preparation needed. CORD-v2 is loaded automatically from HuggingFace at training time via
the `--dataset vlm-hf` flag. The Megatron-Bridge built-in data provider class downloads and caches the dataset on first run.

| Split | Samples |
|---|---|
| Train | 800 |
| Validation | 100 |
| Test | 100 |

Each sample contains a receipt image and a structured ground-truth JSON that is converted
to an XML-like token sequence by `json2token`, e.g.:
```
{"total": {"total_price": "45,500"}}
→ <s_total><s_total_price>45,500</s_total_price></s_total>
```
The model learns to generate this sequence autoregressively given the receipt image.

In [ ]:
# Verify HuggingFace can reach the dataset (or that it is already cached)
from datasets import load_dataset
ds = load_dataset("naver-clova-ix/cord-v2", split="train")
print(f"CORD-v2 train split: {len(ds)} samples")
print(f"Columns: {ds.column_names}")
print(f"Sample ground_truth (first 200 chars): {ds[0]['ground_truth'][:200]}")

---
## Step 3 — Recipe Overview

The recipe `nemotron_omni_cord_v2_peft_config` (in
`/workspace/Megatron-Bridge/src/megatron/bridge/recipes/nemotron_omni/nemotron_omni.py`) configures:

| Setting | Value | Reason |
|---|---|---|
| LoRA target modules | `linear_qkv`, `linear_proj`, `in_proj`, `out_proj` | Attention + Mamba projections |
| LoRA rank / alpha | 16 / 32 | Balanced capacity vs. parameter count |
| Vision encoder | **Frozen** | CORD-v2 is image-only; no video/audio |
| Audio encoder | **Frozen** | No audio in dataset |
| `temporal_patch_dim` | 1 | Disables video temporal patching (image-only) |
| `recompute_granularity` | selective | Saves ~4 GiB activation memory |
| Master weights dtype | fp16 | TE FusedAdam constraint (fp32 or fp16 only) |
| Gradient / moment dtype | bf16 | Fits within 80 GiB H100 with TP=4, EP=8 |

### Parallelism on 8 GPUs (TP=4, EP=8, ETP=1)

| Dimension | Value | Effect |
|---|---|---|
| `tensor_model_parallel_size` | 4 | Dense layers sharded across groups of 4 GPUs |
| `expert_model_parallel_size` | 8 | 128 MoE experts distributed across all 8 GPUs (16 per GPU) |
| `expert_tensor_parallel_size` | 1 | No intra-expert TP sharding |

Dense groups of TP=4 overlap with EP=8 across the same 8 devices, keeping peak GPU memory
within 80 GiB.

---
## Step 4 — Launch Training

In [ ]:
# ── Parallelism ───────────────────────────────────────────────────────────────
N_DEVICES = 8   # GPUs per node
TP        = 4   # Tensor parallelism (dense layers)
EP        = 8   # Expert parallelism (MoE experts — distributed across all GPUs)
ETP       = 1   # Expert tensor parallelism (1 = none)

# ── Training hyperparameters ──────────────────────────────────────────────────
TRAIN_ITERS       = 200
GLOBAL_BATCH_SIZE = 8
MICRO_BATCH_SIZE  = 1
SEQ_LENGTH        = 4096
SAVE_INTERVAL     = 10   # checkpoint every N iters

# ── Output paths ─────────────────────────────────────────────────────────────
OUTPUT_DIR      = "/workspace/results/cord_v2"
CHECKPOINTS_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
TENSORBOARD_DIR = os.path.join(OUTPUT_DIR, "tensorboard")
os.makedirs(OUTPUT_DIR, exist_ok=True)

%env N_DEVICES=$N_DEVICES
%env TP=$TP
%env EP=$EP
%env ETP=$ETP
%env TRAIN_ITERS=$TRAIN_ITERS
%env GLOBAL_BATCH_SIZE=$GLOBAL_BATCH_SIZE
%env MICRO_BATCH_SIZE=$MICRO_BATCH_SIZE
%env SEQ_LENGTH=$SEQ_LENGTH
%env SAVE_INTERVAL=$SAVE_INTERVAL
%env OUTPUT_DIR=$OUTPUT_DIR
%env CHECKPOINTS_DIR=$CHECKPOINTS_DIR
%env TENSORBOARD_DIR=$TENSORBOARD_DIR
print(f"Output : {OUTPUT_DIR}")
print(f"Parallelism : TP={TP}, EP={EP}, ETP={ETP} on {N_DEVICES} GPUs")
print(f"Training    : {TRAIN_ITERS} iters, GBS={GLOBAL_BATCH_SIZE}, MBS={MICRO_BATCH_SIZE}")

In [ ]:
%%bash
set -euo pipefail
export PYTORCH_ALLOC_CONF=expandable_segments:True

echo "=== Training Configuration ==="
echo "  HF model path  : $HF_MODEL_PATH"
echo "  Megatron ckpt  : $MEGATRON_CKPT"
echo "  Checkpoints dir: $CHECKPOINTS_DIR"
echo "  Parallelism    : TP=$TP  EP=$EP  ETP=$ETP"
echo "  Iters / GBS    : $TRAIN_ITERS / $GLOBAL_BATCH_SIZE"
echo ""

cd /workspace/Megatron-Bridge

uv run torchrun --nproc-per-node=$N_DEVICES scripts/training/run_recipe.py \
  --recipe nemotron_omni_cord_v2_peft_config \
  --step_func nemotron_omni_step \
  --dataset vlm-hf \
  checkpoint.pretrained_checkpoint=$MEGATRON_CKPT \
  checkpoint.save=$CHECKPOINTS_DIR \
  checkpoint.load=$CHECKPOINTS_DIR \
  checkpoint.save_interval=$SAVE_INTERVAL \
  checkpoint.finetune=True \
  dataset.hf_processor_path=$HF_MODEL_PATH \
  dataset.trust_remote_code=True \
  model.tensor_model_parallel_size=$TP \
  model.expert_model_parallel_size=$EP \
  model.expert_tensor_parallel_size=$ETP \
  model.seq_length=$SEQ_LENGTH \
  dataset.seq_length=$SEQ_LENGTH \
  train.train_iters=$TRAIN_ITERS \
  train.global_batch_size=$GLOBAL_BATCH_SIZE \
  train.micro_batch_size=$MICRO_BATCH_SIZE \
  logger.tensorboard_dir=$TENSORBOARD_DIR

### Resume Training

To continue from the last saved checkpoint, specify its path in `checkpoint.load` config:

In [ ]:
%%bash
set -euo pipefail
export PYTORCH_ALLOC_CONF=expandable_segments:True

RESUME_ITERS=400   # New total — must be > previous TRAIN_ITERS

cd /workspace/Megatron-Bridge

uv run torchrun --nproc-per-node=$N_DEVICES scripts/training/run_recipe.py \
  --recipe nemotron_omni_cord_v2_peft_config \
  --step_func nemotron_omni_step \
  --dataset vlm-hf \
  checkpoint.load=$CHECKPOINTS_DIR \
  checkpoint.save=$CHECKPOINTS_DIR \
  checkpoint.pretrained_checkpoint=$MEGATRON_CKPT \
  dataset.hf_processor_path=$HF_MODEL_PATH \
  dataset.trust_remote_code=True \
  model.tensor_model_parallel_size=$TP \
  model.expert_model_parallel_size=$EP \
  model.expert_tensor_parallel_size=$ETP \
  model.seq_length=$SEQ_LENGTH \
  dataset.seq_length=$SEQ_LENGTH \
  train.train_iters=$RESUME_ITERS \
  train.global_batch_size=$GLOBAL_BATCH_SIZE \
  train.micro_batch_size=$MICRO_BATCH_SIZE \
  logger.tensorboard_dir=$TENSORBOARD_DIR

---
## Step 5 — Inspect Checkpoint Layout

Megatron Bridge uses PyTorch's distributed checkpoint format (`.distcp`). Each `iter_NNNNNNN`
directory contains one shard file per GPU rank plus shared metadata:

```
$CHECKPOINTS_DIR/
├── iter_0000010/
│   ├── __0_0.distcp        ← rank 0 shard
│   ├── __1_0.distcp        ← rank 1 shard
│   ├── __2_0.distcp
│   ├── __3_0.distcp
│   ├── __4_0.distcp
│   ├── __5_0.distcp
│   ├── __6_0.distcp
│   ├── __7_0.distcp        ← rank 7 shard  (8 shards total for 8 GPUs)
│   ├── common.pt           ← shared state (RNG, iteration counter)
│   ├── metadata.json       ← shard metadata for distributed checkpoint loading
│   ├── run_config.yaml     ← training config snapshot
│   ├── tokenizer/          ← tokenizer files copied from HF model dir
│   └── train_state.pt      ← optimizer and scheduler state
├── iter_0000020/
│   └── …
└── latest_checkpointed_iteration.txt
```

In [ ]:
# Find the latest checkpoint
iter_file = os.path.join(CHECKPOINTS_DIR, "latest_checkpointed_iteration.txt")
assert os.path.exists(iter_file), f"No checkpoint found in {CHECKPOINTS_DIR}. Run training first."

with open(iter_file) as f:
    latest_iter = int(f.read().strip())

LORA_CKPT = os.path.join(CHECKPOINTS_DIR, f"iter_{latest_iter:07d}")
MERGED_CKPT = LORA_CKPT + "_merged"

os.environ["LORA_CKPT"]   = LORA_CKPT
os.environ["MERGED_CKPT"] = MERGED_CKPT

print(f"Latest iteration : {latest_iter}")
print(f"LoRA checkpoint  : {LORA_CKPT}")
print(f"Merge output     : {MERGED_CKPT}")

# List checkpoint shards
shards = sorted(os.listdir(LORA_CKPT)) if os.path.isdir(LORA_CKPT) else []
print(f"Checkpoint shards ({len(shards)}):")
for s in shards[:6]:
    print(f"  {s}")
if len(shards) > 6:
    print(f"  … (+{len(shards)-6} more)")

---
## Step 6 — Export LoRA Adapter

Convert the Megatron PEFT checkpoint into a HuggingFace PEFT adapter directory
(`adapter_config.json` + `adapter_model.safetensors`).

In [ ]:
ADAPTER_DIR = LORA_CKPT + "_hf_adapter"
os.environ["ADAPTER_DIR"] = ADAPTER_DIR
print(f"Adapter output: {ADAPTER_DIR}")

In [ ]:
%%bash
set -euo pipefail
cd /workspace/Megatron-Bridge

uv run python examples/conversion/adapter/export_adapter.py \
    --hf-model-path  "$HF_MODEL_PATH" \
    --lora-checkpoint "$LORA_CKPT" \
    --output         "$ADAPTER_DIR" \
    --trust-remote-code

echo ""
echo "Adapter files:"
ls -lh "$ADAPTER_DIR"

---
## Step 7 — Merge LoRA into Base Megatron Checkpoint

Merge the LoRA adapter weights back into the base Megatron checkpoint to produce a
single dense checkpoint for inference.

In [ ]:
%%bash
set -euo pipefail
cd /workspace/Megatron-Bridge

echo "=== Merge Configuration ==="
echo "  LoRA checkpoint : $LORA_CKPT"
echo "  HF model path   : $HF_MODEL_PATH"
echo "  Merged output   : $MERGED_CKPT"
echo ""

uv run torchrun --nproc-per-node=$N_DEVICES examples/peft/merge_lora.py \
    --lora-checkpoint "$LORA_CKPT" \
    --hf-model-path   "$HF_MODEL_PATH" \
    --output          "$MERGED_CKPT" \
    --tp              $TP

echo ""
echo "Merged checkpoint contents:"
ls "$MERGED_CKPT"

---
## CLI Override Reference

| Override | Description |
|---|---|
| `checkpoint.pretrained_checkpoint=<path>` | Megatron checkpoint to load weights from (first run) |
| `checkpoint.load=<path>` | Directory to resume from (subsequent runs) |
| `checkpoint.save=<path>` | Directory to write new checkpoints |
| `checkpoint.finetune=True` | Start fine-tuning from a pretrained ckpt (skips optimizer resume) |
| `checkpoint.save_interval=10` | Save every N iterations |
| `--dataset vlm-hf` | Use HuggingFace dataset provider |
| `dataset.hf_processor_path=<path>` | HF model dir for tokenizer/processor loading |
| `dataset.trust_remote_code=True` | Required for local HF dirs with custom processor code |
| `dataset.seq_length=4096` | Tokenizer max length (keep in sync with `model.seq_length`) |
| `model.seq_length=4096` | Model context window |
| `model.tensor_model_parallel_size=4` | TP degree for dense layers |
| `model.expert_model_parallel_size=8` | EP degree — distributes MoE experts |
| `model.expert_tensor_parallel_size=1` | TP within each expert (1 = none) |
| `model.recompute_granularity=selective` | Selective activation recompute (~4 GiB savings) |
| `train.train_iters=200` | Total training iterations |
| `train.global_batch_size=8` | Global batch size |
| `train.micro_batch_size=1` | Per-GPU micro batch size |
| `logger.tensorboard_dir=<path>` | TensorBoard log directory |

---

## Comparing with the Nemo Automodel backend

| Aspect | NeMo Automodel | Megatron Bridge |
|---|---|---|
| Parallelism | FSDP2 | TP + EP + ETP (Megatron-Core) |
| Config format | YAML file | Python recipe function |
| Model loading | HF weights directly | HF path (config/tokenizer) + Megatron checkpoint (weights) |
| Data pipeline | HF datasets in-process | `--dataset vlm-hf` or Energon shards |
| Launch | `torchrun -c config.yaml` | `torchrun run_recipe.py --recipe <name> key=value …` |
| Checkpoint format | HF `.safetensors` | Megatron distributed checkpoint |

> **Critical**: `hf_path` in a recipe is used **only** to read `config.json`, the tokenizer,
> and the processor. Model **weights** come exclusively from `checkpoint.pretrained_checkpoint`
> (a Megatron-format directory).
